### WigglNet Keyword Extraction Algorithim

This notebook explains the keyword extraction module used on the WigglNet platform to produce words associated with specific categories.

#### Overview

The `keyword_extractor` module extracts meaningful keywords from text using Natural Language Processing (NLP) techniques. It leverages the NLTK (Natural Language Toolkit) library to identify the most important words in article text.

#### Imports and Preprocessing

First, let's ensure we have the necessary dependencies installed:

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath('../app/article_keywords/tools'))

import nltk
from keyword_extractor import extract_keywords, extract_keywords_from_article

#### How the Algorithm Works

The keyword extraction process follows these steps:

##### 1. Text Preprocessing

- **Input Normalization**: Text is converted to lowercase
- **Tokenization**: Text is split into individual words using NLTK's `word_tokenize()`
- **Stopword Removal**: Common words ("the", "is", "and", etc.) are filtered out
- **Length Filtering**: Only words longer than 3 characters are kept
- **Alphabetic Filtering**: Only alphabetic tokens are retained (no numbers or punctuation)

##### 2. Part-of-Speech (POS) Tagging

The algorithm uses POS tagging to identify the grammatical role of each word:

- **Nouns (NN, NNS, NNP, NNPS)**: Given **3x weight** (most important for keywords)
- **Adjectives (JJ, JJR, JJS)**: Given **2x weight** (moderately important)
- **Other parts of speech**: Given **1x weight**

This weighting ensures that content words (especially nouns) are prioritized over other word types.

##### 3. Frequency Distribution

After weighting:
- A frequency distribution is calculated using NLTK's `FreqDist`
- The most common words (after weighting) are selected

##### 4. Confidence Scoring

Each keyword receives:
- **Frequency**: Raw weighted count
- **Confidence**: Normalized score (0-1) representing the keyword's importance relative to other keywords

Formula: `confidence = keyword_count / total_count_of_all_keywords`

#### Loading Article Data

Using actual files form the /data folder as an example.

In [2]:
import json
import glob

# Find any JSON files in the data folder
data_pattern = '../app/article_keywords/data/*.json'
data_files = glob.glob(data_pattern)

if not data_files:
    raise FileNotFoundError(f"No JSON files found matching pattern: {data_pattern}")

The most recently stored article will be accessed via the `sorted()` method.

In [3]:

# Use the most recent file (sorted by filename, which includes timestamp)
data_file = sorted(data_files)[-1]

with open(data_file, 'r') as f:
    articles = json.load(f)

print(f"Loaded {len(articles)} articles from {data_file}")
print(f"\nFirst article title: {articles[0]['title']}")
print(f"Number of paragraphs: {len(articles[0]['paragraphs'])}")



Loaded 15 articles from ../app/article_keywords/data/html_articles_20260402_150725.json

First article title: 37 Small Bedroom Ideas That Maximize Every Inch
Number of paragraphs: 43


Some high-level information about the pulled article.

In [4]:
# Extract keywords from the first article's paragraphs
article_text = ' '.join(articles[0]['paragraphs'])
keywords = extract_keywords(article_text, num_keywords=10)

# Display results
print("\nExtracted Keywords from First Article:\n")
for i, kw in enumerate(keywords, 1):
    print(f"{i}. {kw['keyword']:20s} | Frequency: {kw['frequency']:3d} | Confidence: {kw['confidence']:.3f}")


Extracted Keywords from First Article:

1. space                | Frequency:  90 | Confidence: 0.205
2. bedroom              | Frequency:  87 | Confidence: 0.198
3. room                 | Frequency:  75 | Confidence: 0.171
4. small                | Frequency:  58 | Confidence: 0.132
5. design               | Frequency:  30 | Confidence: 0.068
6. designer             | Frequency:  24 | Confidence: 0.055
7. home                 | Frequency:  21 | Confidence: 0.048
8. ideas                | Frequency:  18 | Confidence: 0.041
9. feel                 | Frequency:  18 | Confidence: 0.041
10. idea                 | Frequency:  18 | Confidence: 0.041


#### Understanding the Output

Each keyword object contains three fields:

1. **`keyword`**: The extracted word
2. **`frequency`**: The weighted count (after POS weighting)
3. **`confidence`**: A normalized score (0-1) showing relative importance

Notice how domain-specific nouns like "space", "bedroom", "room" rank highly due to:
- High frequency in the text
- Being nouns (3x weight multiplier)
- Not being stopwords

## Comparing Keywords from Different Real Articles

Processing multiple articles from the dataset and applying the keywork extractor is typically how the machine learning models will know where to look to make more accurate predictions.

In [5]:
# Use two different articles from our dataset
article1_text = ' '.join(articles[0]['paragraphs'])
article2_text = ' '.join(articles[1]['paragraphs']) if len(articles) > 1 else article1_text

# Extract keywords from both
article1_keywords = extract_keywords(article1_text, num_keywords=8)
article2_keywords = extract_keywords(article2_text, num_keywords=8)

print(f"ARTICLE 1: {articles[0]['title'][:60]}...")
print("-" * 70)
for kw in article1_keywords:
    print(f"  - {kw['keyword']} (confidence: {kw['confidence']:.3f})")

if len(articles) > 1:
    print(f"\nARTICLE 2: {articles[1]['title'][:60]}...")
    print("-" * 70)
    for kw in article2_keywords:
        print(f"  - {kw['keyword']} (confidence: {kw['confidence']:.3f})")

ARTICLE 1: 37 Small Bedroom Ideas That Maximize Every Inch...
----------------------------------------------------------------------
  - space (confidence: 0.223)
  - bedroom (confidence: 0.216)
  - room (confidence: 0.186)
  - small (confidence: 0.144)
  - design (confidence: 0.074)
  - designer (confidence: 0.060)
  - home (confidence: 0.052)
  - ideas (confidence: 0.045)

ARTICLE 2: 59 Small Patio Ideas That Maximize Your Space...
----------------------------------------------------------------------
  - patio (confidence: 0.234)
  - space (confidence: 0.214)
  - small (confidence: 0.122)
  - outdoor (confidence: 0.115)
  - ideas (confidence: 0.088)
  - home (confidence: 0.085)
  - design (confidence: 0.081)
  - area (confidence: 0.061)


## Stopwords and Filtering

The extractor includes an extended stopword list beyond NLTK's default. This removes common but semantically empty words like:

- **Reporting verbs**: "said", "told", "according"
- **Vague adjectives**: "new", "first", "last"
- **Common verbs**: "would", "could", "may", "get", "make"
- **Generic nouns**: "people", "time", "year"

This ensures only meaningful content words appear in the results.

In [6]:
# Use a real article to demonstrate stopword filtering
# Let's use the first few paragraphs from an article
if len(articles) > 2:
    sample_article = articles[2]
    sample_text = ' '.join(sample_article['paragraphs'][:3])  # First 3 paragraphs
else:
    sample_text = ' '.join(articles[0]['paragraphs'][:3])

keywords_filtered = extract_keywords(sample_text, num_keywords=10)

print("Keywords (after stopword filtering):")
for kw in keywords_filtered:
    print(f"  - {kw['keyword']}")

print("\nNotice that words like 'said', 'told', 'people', 'year' are filtered out!")

Keywords (after stopword filtering):
  - walkway
  - point
  - ideas
  - front
  - path
  - home
  - paths
  - house
  - block
  - door

Notice that words like 'said', 'told', 'people', 'year' are filtered out!


## Using with Article Data

The module provides helper functions for processing article dictionaries:

In [7]:
# Use a real article from our dataset
# Select an article (use the 4th one if available, otherwise the first)
article_index = 3 if len(articles) > 3 else 0
article = articles[article_index]

print(f"Article Title: {article['title']}")
print(f"Source: {article['source']}")
print(f"Number of paragraphs: {len(article['paragraphs'])}\n")

# Extract keywords from article
article_keywords = extract_keywords_from_article(article, num_keywords=10)

print("Article Keywords:")
for kw in article_keywords:
    print(f"  - {kw['keyword']:15s} (freq: {kw['frequency']}, conf: {kw['confidence']:.3f})")

Article Title: These Front Door Colors Instantly Boost Curb Appeal
Source: https://www.housebeautiful.com/room-decorating/
Number of paragraphs: 41

Article Keywords:
  - door            (freq: 94, conf: 0.227)
  - home            (freq: 78, conf: 0.188)
  - front           (freq: 58, conf: 0.140)
  - color           (freq: 45, conf: 0.108)
  - designer        (freq: 36, conf: 0.087)
  - exterior        (freq: 26, conf: 0.063)
  - colors          (freq: 24, conf: 0.058)
  - house           (freq: 18, conf: 0.043)
  - paint           (freq: 18, conf: 0.043)
  - wood            (freq: 18, conf: 0.043)


#### Integration with Scrapers

The keyword extractor is used by both:

1. **`web_scraper.py`** (line 274): Processes HTML articles scraped via Selenium
2. **`rss_scraper.py`** (line 158): Processes articles from RSS feeds

Both scrapers call `add_keywords_to_articles()` which:
- Processes a batch of articles
- Extracts keywords from each article's paragraphs
- Adds a `keywords` field to each article dictionary
- Displays progress and top keywords during processing

#### Key Parameters

`num_keywords`<br><br>
Controls how many keywords to extract (default: 10-15)
- **Lower values** (5-8): Only the most prominent keywords
- **Higher values** (15-20): Broader keyword coverage

### Minimum Word Length
Currently set to 4 characters (line 71 in keyword_extractor.py)
- Filters out short words like "run", "use", "get"
- Keeps more meaningful words like "running", "using", "getting"

In [8]:
# Use a real article to demonstrate num_keywords parameter
article_index = 5 if len(articles) > 5 else 0
sample = ' '.join(articles[article_index]['paragraphs'])

print(f"Using article: {articles[article_index]['title'][:60]}...\n")

print("Top 5 keywords:")
for kw in extract_keywords(sample, num_keywords=5):
    print(f"  - {kw['keyword']}")

print("\nTop 10 keywords:")
for kw in extract_keywords(sample, num_keywords=10):
    print(f"  - {kw['keyword']}")

Using article: The British-Inspired Lighting Your Kitchen NeedsBy Alyssa Kh...

Top 5 keywords:
  - kitchen
  - lamps
  - island
  - design
  - lighting

Top 10 keywords:
  - kitchen
  - lamps
  - island
  - design
  - lighting
  - james
  - riker
  - space
  - room
  - thomas


<b>Algorithm Strengths</b>

1. **Domain-Agnostic**: Works on any topic without training
2. **Fast**: No ML model training required
3. **Transparent**: Clear weighting rules based on POS tags
4. **Customizable**: Easy to adjust stopwords and weights
5. **Confidence Scores**: Provides interpretable relevance metrics

<b>Limitations</b>

1. **No Context Understanding**: Doesn't understand word meanings or relationships
    - Possible solution: Generative AI trained on meanings and relationships
2. **Frequency-Based**: May miss important but infrequent terms
    - Possible soluion: use meaning and relationships to be the bases of searching for outliers like this
3. **Single-Word Only**: Doesn't extract multi-word phrases (e.g., "machine learning")
    - No Solution Yet (Easy)
4. **Language**: Only works for English text
    - No Solution

<b>Potential Improvements</b>

- **N-gram extraction**: Capture phrases like "artificial intelligence"
- **TF-IDF weighting**: Consider word importance across multiple documents
- **Named Entity Recognition**: Specifically extract names, places, organizations
- **Semantic similarity**: Group related keywords together

## Summary

The keyword extractor uses a **rule-based NLP approach** that:

1. Tokenizes and filters text (stopwords, length, alphabetic)
2. Applies POS tagging to identify word types
3. Weights nouns 3x, adjectives 2x, others 1x
4. Calculates frequency distribution on weighted words
5. Returns top keywords with frequency and confidence scores

This approach provides reliable, interpretable keyword extraction suitable for categorizing and indexing scraped articles.